# GAT with Residuals for Mental Health Prediction

## Overview
This notebook implements a **Graph Attention Network (GAT)** with **Residual Connections** to predict student mental health.

### Motivation
Previous analysis showed **low homophily** (friends' mental health correlation ~0.1). A standard GCN, which averages neighbor features, performs poorly in this setting. 

### Key Improvements
1.  **GAT (Graph Attention Network)**: Allows the model to *learn* which friends are relevant and ignore irrelevant ones (assigning low attention weights).
2.  **Residual Connections**: Explicitly adds the node's own features to the layer output (`output = GAT(x) + Linear(x)`). This ensures the model preserves personal characteristics even if the graph structure is noisy.
3.  **Increased Complexity**: Hidden dimension increased to **32**.

---

## 1. Data Loading (Self-Contained)
We implement the data loading logic directly here to ensure the notebook is standalone.

In [11]:
import pandas as pd
import torch
from torch_geometric.data import Data
import numpy as np
import os

# --- Configuration ---
# Paths (Adjusted for where this notebook is located: Code/Analysis/GNN/relationship_reallike)
DATA_DIR = r"../../../../Data/2024data"
RELATIONSHIP_DIR = r"../../../EDA/relationship"

W2_STUDENT_PATH = os.path.join(DATA_DIR, "TIGPS_W2_studentdata_ver5_cleaned_mental_common_only.csv")
OFFLINE_LIKE_PATH = os.path.join(RELATIONSHIP_DIR, "Offline_Like.csv")

# Top 20 Features
NODE_FEATURES = [
    'v51', 'v59_5', 'v52_3', 'v57_4', 'v52_2', 
    'v57_3', 'v50', 'v57_2', 'v52', 'v57_1', 
    'v52_1', 'v57_5', 'v24_2', 'v8_08', 'v28_6', 
    'v39_2', 'v5_5', 'v8_05', 'v24_6', 'v35_2'
]

def load_data_internal():
    print(f"Loading Student Data from: {W2_STUDENT_PATH}")
    try:
        student_df = pd.read_csv(W2_STUDENT_PATH, on_bad_lines='skip', engine='python')
    except Exception as e:
        print(f"Error loading student data: {e}")
        return None

    # --- 1. Prepare Target and Features ---
    mh_cols = [f"v55_{i}" for i in range(1, 15)]
    student_df['mh_score'] = student_df[mh_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
    
    # Filter valid students
    student_df = student_df.dropna(subset=['mh_score', 'student_id']).copy()
    
    # Fill features
    for col in NODE_FEATURES:
        student_df[col] = pd.to_numeric(student_df[col], errors='coerce').fillna(0)
    
    # Create Node Mapping
    unique_student_ids = sorted(student_df['student_id'].unique())
    id_map = {sid: i for i, sid in enumerate(unique_student_ids)}
    num_nodes = len(unique_student_ids)
    print(f"Total Nodes (Students): {num_nodes}")
    
    # Reindex df
    student_df.set_index('student_id', inplace=True)
    student_df = student_df.reindex(unique_student_ids)
    
    x = torch.tensor(student_df[NODE_FEATURES].values, dtype=torch.float)
    y = torch.tensor(student_df['mh_score'].values, dtype=torch.float).view(-1, 1)
    
    # --- 2. Build Mapping for Edges ---
    print("Building ID Mapping dictionary...")
    mapping_dict = {}
    
    def clean_class(val):
        s = str(val).strip()
        if s.endswith('.0'): return s[:-2]
        return s

    student_df['class_clean'] = student_df['class'].apply(clean_class)
    student_df['school_clean'] = pd.to_numeric(student_df['school_id'], errors='coerce').fillna(-1).astype(int)
    student_df['seat_clean'] = pd.to_numeric(student_df['v13'], errors='coerce').fillna(-1).astype(int)
    
    student_df_reset = student_df.reset_index()
    for idx, row in student_df_reset.iterrows():
        sid = row['student_id']
        key = (row['school_clean'], row['class_clean'], row['seat_clean'])
        if row['school_clean'] == -1: continue
        if key not in mapping_dict:
            mapping_dict[key] = sid
    
    # --- 3. Process Edges ---
    print(f"Loading Edges from: {OFFLINE_LIKE_PATH}")
    edges_df = pd.read_csv(OFFLINE_LIKE_PATH)
    
    source_indices = []
    target_indices = []
    
    for _, row in edges_df.iterrows():
        src_sid = row['student_id']
        if src_sid not in id_map: continue
        
        tgt_school = int(pd.to_numeric(row['school_id'], errors='coerce') or -1)
        tgt_class = clean_class(row['class'])
        tgt_seat = int(pd.to_numeric(row['nominated_seat_no'], errors='coerce') or -1)
        
        key = (tgt_school, tgt_class, tgt_seat)
        if key in mapping_dict:
            tgt_sid = mapping_dict[key]
            if tgt_sid in id_map:
                source_indices.append(id_map[src_sid])
                target_indices.append(id_map[tgt_sid])
    
    edge_index = torch.tensor([source_indices, target_indices], dtype=torch.long)
    
    # --- 4. Create Masks (80/10/10) ---
    np.random.seed(42)
    indices = np.random.permutation(num_nodes)
    train_size = int(0.8 * num_nodes)
    val_size = int(0.1 * num_nodes)
    
    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.train_mask[indices[:train_size]] = True
    data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.val_mask[indices[train_size:train_size + val_size]] = True
    data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.test_mask[indices[train_size + val_size:]] = True
    return data

data = load_data_internal()
print("Data Loaded Successfully.")

Loading Student Data from: ../../../../Data/2024data\TIGPS_W2_studentdata_ver5_cleaned_mental_common_only.csv
Total Nodes (Students): 7175
Building ID Mapping dictionary...
Loading Edges from: ../../../EDA/relationship\Offline_Like.csv
Data Loaded Successfully.


## 2. GAT Model Definition

We define `GATResNet` which combines:
1.  **GATConv**: Graph Attention Layer.
2.  **Linear Projection (Residual)**: A parallel linear layer that processes node features independently of the graph.

**Architecture**:
-   **Layer 1**: `ELU(GAT(x) + Linear(x))`
-   **Layer 2**: `GAT(h1) + Linear(h1)` -> Output

Hidden Channels = **32**

In [12]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv, Linear

class GATResNet(torch.nn.Module):
    def __init__(self, num_features, hidden_channels, heads=4):
        super(GATResNet, self).__init__()
        
        # Layer 1
        # The output dimension of GATConv with multi-head is hidden_channels * heads
        self.gat1 = GATConv(num_features, hidden_channels, heads=heads, dropout=0.5)
        self.lin1 = Linear(num_features, hidden_channels * heads)
        
        # Layer 2
        # Input is hidden_channels * heads
        self.gat2 = GATConv(hidden_channels * heads, 1, heads=1, concat=False, dropout=0.5)
        self.lin2 = Linear(hidden_channels * heads, 1)

    def forward(self, x, edge_index):
        # --- First Block ---
        h_graph = self.gat1(x, edge_index)
        h_res = self.lin1(x)
        h = F.elu(h_graph + h_res)
        h = F.dropout(h, p=0.5, training=self.training)
        
        # --- Second Block ---
        out_graph = self.gat2(h, edge_index)
        out_res = self.lin2(h)
        out = out_graph + out_res
        
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GATResNet(num_features=data.num_features, hidden_channels=32, heads=4).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
print(model)

GATResNet(
  (gat1): GATConv(20, 32, heads=4)
  (lin1): Linear(20, 128, bias=True)
  (gat2): GATConv(128, 1, heads=1)
  (lin2): Linear(128, 1, bias=True)
)


## 3. Training Loop (Original 4 Heads)

In [13]:
from sklearn.metrics import r2_score

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.mse_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(mask):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out[mask]
    true = data.y[mask]
    mse = F.mse_loss(pred, true).item()
    r2 = r2_score(true.cpu().numpy(), pred.cpu().numpy())
    return mse, r2

print("Starting Training (4 Heads)...")
for epoch in range(1, 501):
    loss = train()
    if epoch % 50 == 0:
        val_mse, val_r2 = test(data.val_mask)
        print(f"Epoch: {epoch:03d}, Train MSE: {loss:.4f}, Val MSE: {val_mse:.4f}")

test_mse, test_r2 = test(data.test_mask)
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2 Score: {test_r2:.4f}")

Starting Training (4 Heads)...
Epoch: 050, Train MSE: 114.0668, Val MSE: 84.2748
Epoch: 100, Train MSE: 90.0707, Val MSE: 67.5383
Epoch: 150, Train MSE: 81.8719, Val MSE: 60.4613
Epoch: 200, Train MSE: 77.2521, Val MSE: 57.6857
Epoch: 250, Train MSE: 74.9561, Val MSE: 56.1819
Epoch: 300, Train MSE: 72.0563, Val MSE: 55.5072
Epoch: 350, Train MSE: 70.5882, Val MSE: 54.9801
Epoch: 400, Train MSE: 69.7332, Val MSE: 54.5411
Epoch: 450, Train MSE: 67.2971, Val MSE: 54.3109
Epoch: 500, Train MSE: 67.7343, Val MSE: 53.9656
Test MSE: 64.6630
Test R2 Score: 0.4530


## 4. Improvement: Increasing Heads to 8
Here we re-initialize the model with `heads=8` to capture more diverse social signals.
Increasing heads is like having more 'consultants' analyzing the friend network from different perspectives.

In [14]:
# Re-Initialize Model with 8 Heads
print("--- Re-Training with 8 Heads ---")

model_8h = GATResNet(num_features=data.num_features, hidden_channels=32, heads=8).to(device)
optimizer_8h = torch.optim.Adam(model_8h.parameters(), lr=0.005, weight_decay=5e-4)

def train_8h():
    model_8h.train()
    optimizer_8h.zero_grad()
    out = model_8h(data.x, data.edge_index)
    loss = F.mse_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer_8h.step()
    return loss.item()

@torch.no_grad()
def test_8h(mask):
    model_8h.eval()
    out = model_8h(data.x, data.edge_index)
    pred = out[mask]
    true = data.y[mask]
    mse = F.mse_loss(pred, true).item()
    r2 = r2_score(true.cpu().numpy(), pred.cpu().numpy())
    return mse, r2

best_val_r2 = -float('inf')

for epoch in range(1, 501):
    loss = train_8h()
    if epoch % 50 == 0:
        val_mse, val_r2 = test_8h(data.val_mask)
        print(f"[8-Head] Epoch: {epoch:03d}, Train MSE: {loss:.4f}, Val R2: {val_r2:.4f}")
        if val_r2 > best_val_r2:
            best_val_r2 = val_r2

print("--- Final Results (8 Heads) ---")
test_mse_8h, test_r2_8h = test_8h(data.test_mask)
print(f"Test MSE: {test_mse_8h:.4f}")
print(f"Test R2 Score: {test_r2_8h:.4f}")

--- Re-Training with 8 Heads ---
[8-Head] Epoch: 050, Train MSE: 98.7321, Val R2: 0.2342
[8-Head] Epoch: 100, Train MSE: 78.8281, Val R2: 0.3440
[8-Head] Epoch: 150, Train MSE: 73.5422, Val R2: 0.3993
[8-Head] Epoch: 200, Train MSE: 68.9936, Val R2: 0.4227
[8-Head] Epoch: 250, Train MSE: 68.1931, Val R2: 0.4326
[8-Head] Epoch: 300, Train MSE: 66.2260, Val R2: 0.4379
[8-Head] Epoch: 350, Train MSE: 65.6449, Val R2: 0.4400
[8-Head] Epoch: 400, Train MSE: 65.6101, Val R2: 0.4423
[8-Head] Epoch: 450, Train MSE: 63.9795, Val R2: 0.4433
[8-Head] Epoch: 500, Train MSE: 63.7076, Val R2: 0.4457
--- Final Results (8 Heads) ---
Test MSE: 61.9667
Test R2 Score: 0.4758
